# ETAPA 2: Análisis exploratorio y perfilado
# Objetivo: Explorar la evolución de los datos y realizar análisis comparativos.
# Rango temporal: 6 meses de datos (seisMeses–junio 2020).
## Dificultad: Media – combinar múltiples archivos y realizar cálculos agregados. bold text

1. ¿Cuáles son los 10 países con más casos confirmados acumulados durante el semestre?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sb
from IPython.display import display
import datetime

import time
import os
import dask.dataframe as dd

import requests

pd.set_option('display.max_columns', None)

In [ ]:
# Define the start and end dates (6 primeros meses del 2021)
start_date = '2021-01-01'
end_date = '2021-6-30'

date_range = pd.date_range(start=start_date, end=end_date)

df = []
df2 = []

# print(date_range)

In [ ]:
# Importar seisMeses 2021 #2.2

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df.append(pd.read_csv(url))

# Combinar los DataFrames diarios en uno solo.
seisMeses = pd.concat(df, ignore_index=True)

# Calcular tiempo de carga
load_time_1 = time.time() - start_time
print(f"Tiempo de carga: {load_time_1:.2f} s")

In [ ]:
# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df2.append(dd.read_csv(url, dtype={'Admin2': 'object', 'FIPS' : 'object'}, assume_missing=True))

# Combinar los DataFrames diarios en uno solo.
seisMeses = dd.concat(df2, ignore_index=True)

# Calcular tiempo de carga
load_time_2 = time.time() - start_time
print(f"Tiempo de carga con Dask: {load_time_2:.2f} s")

In [ ]:
seisMeses['Province_State'] = seisMeses['Province_State'].astype('category')
seisMeses['Country_Region'] = seisMeses['Country_Region'].astype('category')
seisMeses['Combined_Key'] = seisMeses['Combined_Key'].astype('category')

In [ ]:
seisMeses = seisMeses.drop(columns=['FIPS', 'Admin2', 'Lat', 'Long_', 'Combined_Key'])

In [ ]:
seisMeses = seisMeses.compute()

In [ ]:
totalPaisConfirmados = seisMeses.groupby('Country_Region')['Confirmed'].sum().sort_values(ascending=False)
top10 = totalPaisConfirmados.head(10)
print('Top 10 países con más casos confirmados acumulados al 2020-06-30:')
display(top10.reset_index().rename(columns={'6/30/20':'Confirmado acumulado'}))

2. ¿Qué países presentan mayor tasa de letalidad (Deaths / Confirmed * 100)?

In [ ]:
totalPaisMuertes = seisMeses.groupby('Country_Region')['Deaths'].sum().sort_values(ascending=False)
totalPaisMuertes

In [ ]:
tabla_letalidad = pd.merge(totalPaisConfirmados,totalPaisMuertes, on='Country_Region',how='outer')
tabla_letalidad['tasa_letalidad'] = tabla_letalidad['Confirmed'] / tabla_letalidad['Deaths']
tabla_letalidad
tabla_letalidad = tabla_letalidad.rename(columns={'Confirmed': "Confirmados", 'Deaths': "Muertes"})
tabla_letalidad.sort_values(by='tasa_letalidad', ascending=True)

3. ¿Cuántos países no registran recuperados en los datos analizados?

In [ ]:
#Revisar bien porque cuenta todos los 0, no solo los del recuperado, es una serie
totalRecuperadosPais = seisMeses.groupby('Country_Region')['Recovered'].sum()
NoRecuperados = (totalRecuperadosPais == 0).sum()
print('En total hay ',NoRecuperados,' paises que no registran recuperados en esos 6 meses')

4. ¿Qué país latinoamericano presenta la mayor cantidad de casos activos en junio 2020?

In [ ]:
tablaActivos = pd.merge(tabla_letalidad,totalRecuperadosPais, on='Country_Region',how='outer')
tablaActivos = tablaActivos.rename(columns={'Recovered': "Recuperados"})
tablaActivos['Activos'] = (tablaActivos['Confirmados'] - tablaActivos['Muertes'] - tablaActivos['Recuperados']).clip(lower=0)
mayorActivos = tablaActivos['Activos'].idxmax()
print('El pais latinoamericano con la mayor cantidad de casos activos en junio del 2020 es',mayorActivos)

5. ¿Cómo evolucionaron los casos confirmados en Chile entre seisMeses y junio? (gráfico de
líneas).

In [ ]:
datosChile = seisMeses.loc[seisMeses['Country_Region'] == 'Chile'].copy()
datosChile['Last_Update'] = pd.to_datetime(datosChile['Last_Update'])
datosAgrupados = datosChile.groupby(datosChile['Last_Update'])['Confirmed'].sum().reset_index()
datosAgrupados

plt.figure(figsize=(12, 7))
plt.plot(datosAgrupados['Last_Update'], datosAgrupados['Confirmed'], label='Casos Confirmados', color='b')

plt.title('Evolución de Casos Confirmados en Chile (seisMeses - Junio 2021)', fontsize=16)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Total de Casos Confirmados', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()


6. ¿Cuál fue la fecha con más nuevos casos a nivel mundial durante este período?

In [ ]:
casosNuevos = seisMeses[['Country_Region','Last_Update','Confirmed']].copy()
casosNuevos['Last_Update'] = pd.to_datetime(casosNuevos['Last_Update'], format='ISO8601')

casosNuevos['Last_Update'] = casosNuevos['Last_Update'].dt.date

casosNuevos_mundial = casosNuevos.groupby('Last_Update')['Confirmed'].sum().reset_index()
casosNuevos_mundial['Nuevos Casos'] = casosNuevos_mundial['Confirmed'].diff()

fila_max_nuevos = casosNuevos_mundial.loc[casosNuevos_mundial['Nuevos Casos'].idxmax()]

fecha_max = fila_max_nuevos['Last_Update']
casos_max = fila_max_nuevos['Nuevos Casos']

print(f"\nLa fecha con más nuevos casos reportados a nivel mundial fue: {fecha_max}, con un total de {int(casos_max):,} casos nuevos")

7. ¿Existe correlación entre casos confirmados y fallecidos? (gráfico de dispersión +
regresión).

In [ ]:
cols_to_plot = ['Confirmed', 'Deaths']
seisMeses_paises = seisMeses.copy()

In [ ]:
plt.figure(figsize=(12, 7))

# Usamos sns.scatterplot (en lugar de regplot)
# 'alpha=0.1' hace los puntos 90% transparentes para ver la densidad
# 's=10' hace los puntos más pequeños
sb.scatterplot(
    data=seisMeses_paises, 
    x='Confirmed', 
    y='Deaths',
    alpha=0.1, 
    s=10,
    edgecolor='none' # Quitar bordes de los puntos
)

# ¡¡Crucial!! Usar escala logarítmica para que los datos se separen
plt.xscale('log')
plt.yscale('log')

plt.title('Dispersión de Casos Confirmados vs. Muertes (Registros Diarios)', fontsize=16)
plt.xlabel('Casos Confirmados (Escala Logarítmica)', fontsize=12)
plt.ylabel('Muertes (Escala Logarítmica)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5, which='both') 
plt.tight_layout()

In [ ]:
plt.figure(figsize=(12, 7))

# sns.regplot() dibuja los puntos Y la línea de regresión
sb.regplot(
    data=seisMeses_paises, 
    x='Confirmed', 
    y='Deaths',
    scatter_kws={'alpha': 0.5, 's': 50},       # Estilo de los puntos
    line_kws={'color': 'red', 'linestyle': '--'} # Estilo de la línea
)

# ¡¡Crucial!! Usar escala logarítmica en ambos ejes
plt.xscale('log')
plt.yscale('log')

plt.title('Regresión de Muertes vs. Casos Confirmados (Totales por País)', fontsize=16)
plt.xlabel('Total Casos Confirmados (Escala Logarítmica)', fontsize=12)
plt.ylabel('Total Muertes (Escala Logarítmica)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5, which='both')
plt.tight_layout()

# Guardar el Gráfico
output_filename = 'grafico_regresion_por_pais.png'
plt.savefig(output_filename)
print(f"Gráfico guardado exitosamente como: {output_filename}")

8. Mostrar el Top 10 de países con mayor crecimiento porcentual de casos entre mayo y
junio.

In [ ]:
mayorCrecimiento = seisMeses.copy()
mayorCrecimiento['Last_Update'] = pd.to_datetime(mayorCrecimiento['Last_Update'], format='ISO8601')
crecimientoDaily = mayorCrecimiento.groupby(['Country_Region', 'Last_Update'])['Confirmed'].sum().reset_index()

fecha_mayo = pd.to_datetime('2021-05-31')
fecha_junio = pd.to_datetime('2021-06-30')

#Crecimiento historico finales de mayo.
df_mayo_historico = crecimientoDaily[crecimientoDaily['Last_Update'] <= fecha_mayo]
df_mayo_final = df_mayo_historico.groupby('Country_Region')['Confirmed'].max().reset_index()
df_mayo_final.columns = ['Country_Region', 'Total_Mayo']

#crecimiento historico finales de junio
df_junio_historico = crecimientoDaily[crecimientoDaily['Last_Update'] <= fecha_junio]
df_junio_final = df_junio_historico.groupby('Country_Region')['Confirmed'].max().reset_index()
df_junio_final.columns = ['Country_Region', 'Total_Junio']

crecimientoTotal = pd.merge(df_mayo_final, df_junio_final, on='Country_Region', how='inner')
threshold = 100
df_growth_filtrado = crecimientoTotal[crecimientoTotal['Total_Mayo'] > threshold]

df_growth_filtrado['Crecimiento_Porc'] = ((df_growth_filtrado['Total_Junio'] - df_growth_filtrado['Total_Mayo']) / df_growth_filtrado['Total_Mayo']) * 100
df_top10 = df_growth_filtrado.sort_values(by='Crecimiento_Porc', ascending=False).head(10)
df_top10


9. Identificar países con rebrote (un día sin casos y luego un incremento posterior).

In [ ]:
rebrote = seisMeses.copy()
rebrote['Last_Update'] = pd.to_datetime(rebrote['Last_Update'], format='ISO8601')

rebrote = rebrote.sort_values(by=['Country_Region', 'Last_Update'])
rebrote['Nuevos_Casos'] = rebrote.groupby('Country_Region')['Confirmed'].diff()

rebrote['Nuevos_Casos'] = rebrote['Nuevos_Casos'].fillna(rebrote['Confirmed'])
rebrote['Nuevos_Casos'] = rebrote['Nuevos_Casos'].clip(lower=0)

In [ ]:
paises_con_rebrote = []
datos_agrupados = rebrote.groupby('Country_Region')
for nombre_pais, datos_pais in datos_agrupados:
    
    # Obtener todas las fechas donde hubo 0 casos nuevos
    fechas_cero_casos = datos_pais[datos_pais['Nuevos_Casos'] == 0]['Last_Update']
    
    # Si nunca tuvo un día con 0 casos, no puede tener rebrote
    if fechas_cero_casos.empty:
        continue
        
    # Encontrar la PRIMERA fecha con 0 casos
    primera_fecha_cero = fechas_cero_casos.min()
    
    # Buscar si existe CUALQUIER día POSTERIOR con casos > 0
    tiene_crecimiento_posterior = datos_pais[
        (datos_pais['Last_Update'] > primera_fecha_cero) & 
        (datos_pais['Nuevos_Casos'] > 0)
    ].any().any() # .any().any() comprueba si hay algún True en el sub-dataframe
    
    if tiene_crecimiento_posterior:
        paises_con_rebrote.append(nombre_pais)

# --- 5. Mostrar Resultados ---

print("\n--- Países Identificados con Rebrote ---")
if paises_con_rebrote:
    for pais in paises_con_rebrote:
        print(f"- {pais}")
else:
    print("No se encontraron países con el patrón de rebrote especificado.")

10. Generar un reporte de perfilado automático (ydata-profiling o pandas_profiling) que incluya
distribuciones, correlaciones y resumen de calidad de datos.

In [1]:
# Instalar la librería si no está disponible (uso de magic %pip en Jupyter)
%pip install -q ydata-profiling

^C
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/__main__.py", line 22, in <module>
    from pip._internal.cli.main import main as _main
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/cli/autocompletion.py", line 12, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spinners import open_spinner
  

In [ ]:
from ydata_profiling import ProfileReport

# Crea el objeto de reporte
profile = ProfileReport(seisMeses, title="Reporte de Perfilado - Datos COVID",
                        explorative=False,
                        minimal=True,
                        )


# --- 3. Guardar el Reporte como HTML ---
output_filename = "perfilado.html"

try:
    os.remove(output_filename)
    print(f"Archivo existente '{output_filename}' eliminado.")
except FileNotFoundError:
    pass

profile.to_file(output_filename)
print("--- ¡Reporte Generado! ---")